In [1]:
# Keep Colab alive
import time
from IPython.display import display, HTML

display(HTML("""
<script>
function ClickConnect(){
  console.log("Keeping Colab alive");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
</script>
"""))
print("✅ Keep-alive script running. Close this cell when done.")

✅ Keep-alive script running. Close this cell when done.


In [ ]:
!pip install ultralytics

In [4]:
"""
Reproducibility Script for Ganoderma Detection Experiment
- Reads pre-saved fold splits
- Creates symlinks on the fly
- Trains baseline and improved models
- Evaluates on test sets
"""

import os
import random
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from ultralytics import YOLO

# ============================================================
# CONFIGURATION
# ============================================================

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
BASE_DIR = "/content/ganoderma"
TRAIN_IMG_DIR = f"{BASE_DIR}/train/images"
TRAIN_LABEL_DIR = f"{BASE_DIR}/train/labels"
VALID_IMG_DIR = f"{BASE_DIR}/valid/images"
VALID_LABEL_DIR = f"{BASE_DIR}/valid/labels"
TEST_IMG_DIR = f"{BASE_DIR}/test/images"
TEST_LABEL_DIR = f"{BASE_DIR}/test/labels"

# Split files location (from backup)
SPLITS_DIR = "/content/drive/MyDrive/experiment_backup/splits"

# Training parameters
EPOCHS = 50
BATCH_SIZE = 30
IMGSZ = 640
DEVICE = 0

# ============================================================
# 1. RESTORE DATASET
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive if not exists
if not os.path.exists(BASE_DIR):
    print("Copying dataset from Drive...")
    !cp -r "/content/drive/MyDrive/ganoderma" /content/ganoderma
    print("✅ Dataset restored")

print(f"Dataset location: {BASE_DIR}")

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def get_source_dir(img):
    """Return source directories for an image"""
    if os.path.exists(os.path.join(TRAIN_IMG_DIR, img)):
        return TRAIN_IMG_DIR, TRAIN_LABEL_DIR
    elif os.path.exists(os.path.join(VALID_IMG_DIR, img)):
        return VALID_IMG_DIR, VALID_LABEL_DIR
    else:
        return TEST_IMG_DIR, TEST_LABEL_DIR


def create_symlinks(images, dst_img_dir, dst_label_dir):
    """Create symbolic links to images and labels"""
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_label_dir, exist_ok=True)

    for img in images:
        src_img, src_label = get_source_dir(img)
        dst_img = os.path.join(dst_img_dir, img)
        if not os.path.exists(dst_img):
            os.symlink(os.path.join(src_img, img), dst_img)

        label = img.replace('.jpg', '.txt').replace('.png', '.txt')
        src_label_path = os.path.join(src_label, label)
        dst_label = os.path.join(dst_label_dir, label)
        if os.path.exists(src_label_path) and not os.path.exists(dst_label):
            os.symlink(src_label_path, dst_label)


def clear_folders(fold_dir):
    """Remove existing symlink folders"""
    for subdir in ['train', 'val', 'test', 'improved_train', 'improved_val']:
        for subsub in ['images', 'labels']:
            path = f"{fold_dir}/{subdir}/{subsub}"
            if os.path.exists(path):
                shutil.rmtree(path)


def read_split_file(fold, set_type):
    """Read image list from split file"""
    split_file = f"{SPLITS_DIR}/fold_{fold}_{set_type}.txt"
    if os.path.exists(split_file):
        with open(split_file, 'r') as f:
            return [line.strip() for line in f.readlines() if line.strip()]
    return None


def find_hard_examples(model, val_img_dir, val_label_dir, data_yaml):
    """Find hard examples using model predictions"""
    val_results = model.val(data=data_yaml, save_txt=True)
    pred_dir = val_results.save_dir

    hard_examples = set()

    for label_file in os.listdir(val_label_dir):
        if not label_file.endswith('.txt'):
            continue

        img_name = label_file.replace('.txt', '.jpg')

        with open(os.path.join(val_label_dir, label_file), 'r') as f:
            gt_count = len(f.readlines())

        pred_file = os.path.join(pred_dir, label_file)
        pred_count = 0
        if os.path.exists(pred_file):
            with open(pred_file, 'r') as f:
                pred_count = len(f.readlines())

        if gt_count > 0 and pred_count == 0:
            hard_examples.add(img_name)
        elif gt_count == 0 and pred_count > 0:
            hard_examples.add(img_name)
        elif gt_count > 0 and pred_count != gt_count:
            hard_examples.add(img_name)

    return hard_examples


# ============================================================
# 3. RUN EXPERIMENT FOR ALL FOLDS
# ============================================================

print("\n" + "="*60)
print("RUNNING REPRODUCIBILITY EXPERIMENT")
print("="*60)

results = []

for fold in range(1, 6):
    print(f"\n{'='*50}")
    print(f"FOLD {fold}/5")
    print(f"{'='*50}")

    # Read split files
    train_subset = read_split_file(fold, 'train')
    val_subset = read_split_file(fold, 'val')
    test_files = read_split_file(fold, 'test')

    if train_subset is None or val_subset is None or test_files is None:
        print(f"  ❌ Split files for Fold {fold} not found. Skipping.")
        continue

    print(f"  Train: {len(train_subset)} images")
    print(f"  Val: {len(val_subset)} images")
    print(f"  Test: {len(test_files)} images")

    # Create fold directory
    fold_dir = f"/content/fold_{fold}"
    os.makedirs(fold_dir, exist_ok=True)

    # Clear old symlinks
    clear_folders(fold_dir)

    # Create symlinks for initial data
    train_img_dir = f"{fold_dir}/train/images"
    train_label_dir = f"{fold_dir}/train/labels"
    val_img_dir = f"{fold_dir}/val/images"
    val_label_dir = f"{fold_dir}/val/labels"
    test_img_dir = f"{fold_dir}/test/images"
    test_label_dir = f"{fold_dir}/test/labels"

    create_symlinks(train_subset, train_img_dir, train_label_dir)
    create_symlinks(val_subset, val_img_dir, val_label_dir)
    create_symlinks(test_files, test_img_dir, test_label_dir)

    # ============================================================
    # BASELINE TRAINING
    # ============================================================

    print(f"\n  📊 Training baseline model...")
    baseline_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
nc: 1
names: ['Ganoderma']
"""
    with open(f"{fold_dir}/baseline_data.yaml", 'w') as f:
        f.write(baseline_yaml)

    model = YOLO('yolov8n.pt')
    baseline_results = model.train(
        data=f"{fold_dir}/baseline_data.yaml",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=fold_dir,
        name="baseline",
        exist_ok=True,
        verbose=False
    )

    baseline_val = model.val(data=f"{fold_dir}/baseline_data.yaml")
    baseline_score = baseline_val.box.map50
    print(f"  Baseline mAP@50: {baseline_score:.4f}")

    # ============================================================
    # FIND HARD EXAMPLES
    # ============================================================

    print(f"\n  🔍 Finding hard examples...")
    hard_examples = find_hard_examples(model, val_img_dir, val_label_dir, f"{fold_dir}/baseline_data.yaml")
    hard_examples = hard_examples & set(val_subset)
    print(f"  Hard examples found: {len(hard_examples)}")

    # ============================================================
    # SWAP HARD EXAMPLES
    # ============================================================

    if len(hard_examples) == 0:
        print("  No hard examples found. Using baseline model.")
        improved_score = baseline_score
        best_model_path = f"{fold_dir}/baseline/weights/best.pt"
        selected = 'baseline'
        new_train_list = train_subset
        new_val_list = val_subset
    else:
        print(f"\n  📊 SWAPPING {len(hard_examples)} hard examples...")

        new_val_set = set(val_subset) - hard_examples
        current_train_set = set(train_subset)
        train_not_hard = list(current_train_set - hard_examples)
        random.shuffle(train_not_hard)
        images_to_remove = set(train_not_hard[:len(hard_examples)])

        new_train_set = (current_train_set - images_to_remove) | hard_examples
        new_val_set = new_val_set | images_to_remove

        new_train_list = list(new_train_set)
        new_val_list = list(new_val_set)

        print(f"    Training: {len(train_subset)} → {len(new_train_list)}")
        print(f"    Validation: {len(val_subset)} → {len(new_val_list)}")

        # Create improved symlinks
        improved_train_img_dir = f"{fold_dir}/improved_train/images"
        improved_train_label_dir = f"{fold_dir}/improved_train/labels"
        improved_val_img_dir = f"{fold_dir}/improved_val/images"
        improved_val_label_dir = f"{fold_dir}/improved_val/labels"

        for d in [improved_train_img_dir, improved_train_label_dir, improved_val_img_dir, improved_val_label_dir]:
            if os.path.exists(d):
                shutil.rmtree(d)

        create_symlinks(new_train_list, improved_train_img_dir, improved_train_label_dir)
        create_symlinks(new_val_list, improved_val_img_dir, improved_val_label_dir)

        # ============================================================
        # IMPROVED MODEL TRAINING
        # ============================================================

        print(f"\n  📊 Training improved model from scratch...")
        improved_yaml = f"""
train: {improved_train_img_dir}
val: {improved_val_img_dir}
nc: 1
names: ['Ganoderma']
"""
        with open(f"{fold_dir}/improved_data.yaml", 'w') as f:
            f.write(improved_yaml)

        improved_model = YOLO('yolov8n.pt')
        improved_results = improved_model.train(
            data=f"{fold_dir}/improved_data.yaml",
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH_SIZE,
            device=DEVICE,
            project=fold_dir,
            name="improved",
            exist_ok=True,
            verbose=False
        )

        improved_val = improved_model.val(data=f"{fold_dir}/improved_data.yaml")
        improved_score = improved_val.box.map50
        improved_model_path = f"{fold_dir}/improved/weights/best.pt"

        print(f"  Improved mAP@50: {improved_score:.4f}")

        if improved_score > baseline_score:
            best_model_path = improved_model_path
            best_score = improved_score
            selected = 'improved'
            print(f"  ✅ Improved model selected (+{improved_score - baseline_score:.4f})")
        else:
            best_model_path = f"{fold_dir}/baseline/weights/best.pt"
            best_score = baseline_score
            selected = 'baseline'
            print(f"  ➡️ Baseline model selected")

    # ============================================================
    # TEST ON HELD-OUT FOLD
    # ============================================================

    print(f"\n  📊 Testing on held-out fold...")
    best_model = YOLO(best_model_path)

    test_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
test: {test_img_dir}
nc: 1
names: ['Ganoderma']
"""
    with open(f"{fold_dir}/test_data.yaml", 'w') as f:
        f.write(test_yaml)

    test_results = best_model.val(data=f"{fold_dir}/test_data.yaml", split='test')

    results.append({
        'fold': fold,
        'baseline_score': baseline_score,
        'improved_score': improved_score if len(hard_examples) > 0 else baseline_score,
        'test_map': test_results.box.map50,
        'test_precision': test_results.box.mp,
        'test_recall': test_results.box.mr,
        'hard_examples': len(hard_examples),
        'selected': selected
    })

    print(f"  Test mAP@50: {test_results.box.map50:.4f}")
    print(f"  Test Precision: {test_results.box.mp:.4f}")
    print(f"  Test Recall: {test_results.box.mr:.4f}")

# ============================================================
# 4. FINAL RESULTS
# ============================================================

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

results_df = pd.DataFrame(results)
print("\nPer-fold results:")
print(results_df.to_string(index=False))

print("\n" + "-"*60)
print("SUMMARY STATISTICS:")
print("-"*60)

maps = [r['test_map'] for r in results]
precisions = [r['test_precision'] for r in results]
recalls = [r['test_recall'] for r in results]

print(f"Test mAP@50:  {np.mean(maps):.4f} ± {np.std(maps):.4f}")
print(f"Test Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"Test Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")

# Save results
results_df.to_csv("/content/reproduced_results.csv", index=False)
print("\n✅ Results saved to: reproduced_results.csv")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
Copying dataset from Drive...
✅ Dataset restored
Dataset location: /content/ganoderma

RUNNING REPRODUCIBILITY EXPERIMENT

FOLD 1/5
  ❌ Split files for Fold 1 not found. Skipping.

FOLD 2/5
  ❌ Split files for Fold 2 not found. Skipping.

FOLD 3/5
  ❌ Split files for Fold 3 not found. Skipping.

FOLD 4/5
  ❌ Split files for Fold 4 not found. Skipping.

FOLD 5/5
  ❌ Split files for Fold 5 not found. Skipping.

FINAL RESULTS

Per-fold results:
Empty DataFrame
Columns: []
Index: []

------------------------------------------------------------
SUMMARY STATISTICS:
------------------------------------------------------------
Test mAP@50:  nan ± nan
Test Precisi

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [5]:
"""
Generate 500 Synthetic Images
Patches from diseased → pasted onto DIFFERENT healthy images
"""

import os
import shutil
import random
import cv2
import numpy as np
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# RESTORE DATASET
# ============================================================
print("="*60)
print("Step 1: Restoring dataset")
print("="*60)

if not os.path.exists("/content/ganoderma"):
    !cp -r "/content/drive/MyDrive/ganoderma" /content/ganoderma
print("✅ Dataset restored")

# ============================================================
# COLLECT DISEASED AND HEALTHY IMAGES
# ============================================================
print("\n" + "="*60)
print("Step 2: Collecting images")
print("="*60)

DISEASED_IMAGES = []
HEALTHY_IMAGES = []

train_img_dir = "/content/ganoderma/train/images"
train_label_dir = "/content/ganoderma/train/labels"

for img_name in os.listdir(train_img_dir):
    if not img_name.endswith(('.jpg', '.png')):
        continue

    label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
    label_path = os.path.join(train_label_dir, label_name)
    img_path = os.path.join(train_img_dir, img_name)

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            if len(f.readlines()) > 0:
                DISEASED_IMAGES.append(img_path)
                continue

    HEALTHY_IMAGES.append(img_path)

print(f"Diseased images: {len(DISEASED_IMAGES)}")
print(f"Healthy images: {len(HEALTHY_IMAGES)}")

# ============================================================
# EXTRACT PATCHES FROM DISEASED IMAGES
# ============================================================
print("\n" + "="*60)
print("Step 3: Extracting disease patches")
print("="*60)

def extract_patches(img_path):
    """Extract all disease patches from an image"""
    img = cv2.imread(img_path)
    if img is None:
        return []

    h, w = img.shape[:2]
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')

    patches = []

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    x_c, y_c, box_w, box_h = map(float, parts[1:5])
                    x1 = int((x_c - box_w/2) * w)
                    y1 = int((y_c - box_h/2) * h)
                    x2 = int((x_c + box_w/2) * w)
                    y2 = int((y_c + box_h/2) * h)

                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)

                    if x2 > x1 and y2 > y1:
                        patch = img[y1:y2, x1:x2]
                        if patch.size > 0:
                            patches.append(patch)

    return patches

ALL_PATCHES = []
for img_path in tqdm(DISEASED_IMAGES, desc="Extracting"):
    patches = extract_patches(img_path)
    ALL_PATCHES.extend(patches)

print(f"Total patches extracted: {len(ALL_PATCHES)}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def match_histogram(source, target):
    """Match source histogram to target"""
    source_lab = cv2.cvtColor(source, cv2.COLOR_BGR2LAB)
    target_lab = cv2.cvtColor(target, cv2.COLOR_BGR2LAB)

    for i in range(3):
        source_ch = source_lab[:,:,i]
        target_ch = target_lab[:,:,i]

        source_hist, _ = np.histogram(source_ch.flatten(), 256, [0,256])
        target_hist, _ = np.histogram(target_ch.flatten(), 256, [0,256])

        source_cdf = source_hist.cumsum() / source_hist.sum()
        target_cdf = target_hist.cumsum() / target_hist.sum()

        lookup = np.zeros(256, dtype=np.uint8)
        for j in range(256):
            lookup[j] = np.argmin(np.abs(source_cdf[j] - target_cdf))

        source_lab[:,:,i] = cv2.LUT(source_ch, lookup)

    return cv2.cvtColor(source_lab, cv2.COLOR_LAB2BGR)


def feather_mask(mask, kernel_size=15):
    if kernel_size % 2 == 0:
        kernel_size += 1
    return cv2.GaussianBlur(mask, (kernel_size, kernel_size), 0)


def rotate_patch(patch, angle):
    h, w = patch.shape[:2]
    center = (w//2, h//2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(patch, M, (w, h), flags=cv2.INTER_LANCZOS4)


def paste_patch(bg, patch):
    """Paste patch onto background"""
    patch_matched = match_histogram(patch, bg)

    h, w = patch_matched.shape[:2]
    bg_h, bg_w = bg.shape[:2]

    if h > bg_h or w > bg_w:
        return bg, None

    y = random.randint(0, max(0, bg_h - h))
    x = random.randint(0, max(0, bg_w - w))

    mask = cv2.cvtColor(patch_matched, cv2.COLOR_BGR2GRAY)
    mask = (mask > 10).astype(np.uint8) * 255
    mask = feather_mask(mask)
    mask = mask / 255.0

    for c in range(3):
        bg[y:y+h, x:x+w, c] = (
            patch_matched[:,:,c] * mask +
            bg[y:y+h, x:x+w, c] * (1 - mask)
        )

    return bg, (x, y, x+w, y+h)


# ============================================================
# GENERATE 500 SYNTHETIC IMAGES
# ============================================================
print("\n" + "="*60)
print("Step 4: Generating 500 synthetic images")
print("   Patches from diseased → pasted onto DIFFERENT healthy")
print("="*60)

SYNTH_DIR = "/content/synthetic_final"

# Clear old folder
if os.path.exists(SYNTH_DIR):
    shutil.rmtree(SYNTH_DIR)
    print("✅ Cleared old synthetic folder")

os.makedirs(f"{SYNTH_DIR}/images", exist_ok=True)
os.makedirs(f"{SYNTH_DIR}/labels", exist_ok=True)

NUM_SYNTHETIC = 500
successful = 0

for i in tqdm(range(NUM_SYNTHETIC), desc="Generating"):
    # Pick random healthy background
    bg_path = random.choice(HEALTHY_IMAGES)
    bg = cv2.imread(bg_path)
    if bg is None:
        continue

    bg_h, bg_w = bg.shape[:2]
    num_patches = random.randint(1, 2)
    boxes = []
    attempts = 0

    while len(boxes) < num_patches and attempts < 20:
        patch = random.choice(ALL_PATCHES).copy()

        # Augment
        if random.random() > 0.5:
            patch = cv2.flip(patch, 1)

        if random.random() > 0.6:
            angle = random.uniform(-15, 15)
            patch = rotate_patch(patch, angle)

        if patch.shape[0] > bg_h or patch.shape[1] > bg_w:
            attempts += 1
            continue

        bg, box = paste_patch(bg, patch)
        if box is not None:
            boxes.append(box)

        attempts += 1

    if len(boxes) == 0:
        continue

    out_name = f"synth_{i:06d}.jpg"
    cv2.imwrite(os.path.join(SYNTH_DIR, "images", out_name), bg)

    with open(os.path.join(SYNTH_DIR, "labels", out_name.replace('.jpg', '.txt')), 'w') as f:
        for box in boxes:
            x1, y1, x2, y2 = box
            xc = (x1 + x2) / 2 / bg_w
            yc = (y1 + y2) / 2 / bg_h
            bw = (x2 - x1) / bg_w
            bh = (y2 - y1) / bg_h
            f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    successful += 1

print(f"\n✅ Generated {successful} synthetic images")
print(f"   Saved to: {SYNTH_DIR}")

# Save to Drive
!cp -r /content/synthetic_final /content/drive/MyDrive/synthetic_final
print("✅ Saved to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Restoring dataset
✅ Dataset restored

Step 2: Collecting images
Diseased images: 600
Healthy images: 387

Step 3: Extracting disease patches


Extracting: 100%|██████████| 600/600 [00:02<00:00, 247.45it/s]


Total patches extracted: 1215

Step 4: Generating 500 synthetic images
   Patches from diseased → pasted onto DIFFERENT healthy


Generating: 100%|██████████| 500/500 [00:27<00:00, 18.15it/s]



✅ Generated 500 synthetic images
   Saved to: /content/synthetic_final
✅ Saved to Drive


In [6]:
"""
Script 2: Create Master Dataset and Generate 5-Fold Splits
FIXED: Synthetic images only in training
"""

import os
import random
import shutil
from sklearn.model_selection import KFold
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# RESTORE DATA
# ============================================================
print("="*60)
print("Step 1: Restoring data")
print("="*60)

if not os.path.exists("/content/ganoderma"):
    !cp -r "/content/drive/MyDrive/ganoderma" /content/ganoderma

if not os.path.exists("/content/synthetic_final") and os.path.exists("/content/drive/MyDrive/synthetic_final"):
    !cp -r "/content/drive/MyDrive/synthetic_final" /content/synthetic_final

print("✅ Data restored")

# ============================================================
# COLLECT ALL ORIGINAL IMAGES (ONLY)
# ============================================================
print("\n" + "="*60)
print("Step 2: Collecting all original images")
print("="*60)

ALL_IMAGES = []
for img in os.listdir("/content/ganoderma/train/images"):
    if img.endswith(('.jpg', '.png')):
        ALL_IMAGES.append(img)
for img in os.listdir("/content/ganoderma/valid/images"):
    if img.endswith(('.jpg', '.png')):
        ALL_IMAGES.append(img)
for img in os.listdir("/content/ganoderma/test/images"):
    if img.endswith(('.jpg', '.png')):
        ALL_IMAGES.append(img)

print(f"Original images: {len(ALL_IMAGES)}")

# ============================================================
# ADD SYNTHETIC IMAGES (SEPARATE POOL)
# ============================================================
print("\n" + "="*60)
print("Step 3: Loading synthetic images")
print("="*60)

SYNTH_IMAGES = []
if os.path.exists("/content/synthetic_final/images"):
    SYNTH_IMAGES = [f for f in os.listdir("/content/synthetic_final/images") if f.endswith(('.jpg', '.png'))]
    print(f"Synthetic images: {len(SYNTH_IMAGES)}")
else:
    print("❌ Synthetic images not found! Run generation script first.")
    exit()

# ============================================================
# CREATE MASTER DATASET
# ============================================================
print("\n" + "="*60)
print("Step 4: Creating master dataset")
print("="*60)

MASTER_DIR = "/content/master_dataset"
if os.path.exists(MASTER_DIR):
    shutil.rmtree(MASTER_DIR)

os.makedirs(f"{MASTER_DIR}/images", exist_ok=True)
os.makedirs(f"{MASTER_DIR}/labels", exist_ok=True)

# Copy original images
print("Copying original images...")
for img_name in ALL_IMAGES:
    for src in ["train", "valid", "test"]:
        src_img = f"/content/ganoderma/{src}/images/{img_name}"
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(MASTER_DIR, "images", img_name))
            break
    label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
    for src in ["train", "valid", "test"]:
        src_label = f"/content/ganoderma/{src}/labels/{label_name}"
        if os.path.exists(src_label):
            shutil.copy(src_label, os.path.join(MASTER_DIR, "labels", label_name))
            break

# Copy synthetic images
print("Copying synthetic images...")
for img_name in SYNTH_IMAGES:
    shutil.copy(os.path.join("/content/synthetic_final/images", img_name), os.path.join(MASTER_DIR, "images", img_name))
    label_name = img_name.replace('.jpg', '.txt')
    shutil.copy(os.path.join("/content/synthetic_final/labels", label_name), os.path.join(MASTER_DIR, "labels", label_name))

master_images = os.listdir(f"{MASTER_DIR}/images")
print(f"\n✅ Master dataset created with {len(master_images)} images")

# ============================================================
# SEPARATE ORIGINAL AND SYNTHETIC
# ============================================================
print("\n" + "="*60)
print("Step 5: Separating original and synthetic")
print("="*60)

original_only = [img for img in master_images if not img.startswith("synth_")]
synthetic_only = [img for img in master_images if img.startswith("synth_")]

print(f"Original images only: {len(original_only)}")
print(f"Synthetic images only: {len(synthetic_only)}")

# ============================================================
# GENERATE 5-FOLD SPLITS USING ONLY ORIGINAL IMAGES
# ============================================================
print("\n" + "="*60)
print("Step 6: Generating 5-fold splits (original images only)")
print("="*60)

random.seed(42)
original_list = original_only.copy()
random.shuffle(original_list)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_folds = []

for fold, (train_idx, test_idx) in enumerate(kf.split(original_list)):
    train_original = [original_list[i] for i in train_idx]
    test_original = [original_list[i] for i in test_idx]

    # Split train into train/val (75/25)
    random.shuffle(train_original)
    split_idx = int(0.75 * len(train_original))
    train_original_subset = train_original[:split_idx]
    val_original_subset = train_original[split_idx:]

    # ADD synthetic images ONLY to training set
    train_all = train_original_subset + synthetic_only
    val_all = val_original_subset
    test_all = test_original

    # Shuffle training set
    random.shuffle(train_all)

    all_folds.append({
        'fold': fold + 1,
        'train': train_all,
        'val': val_all,
        'test': test_all
    })

    print(f"Fold {fold+1}: Train={len(train_all)} (orig={len(train_original_subset)} + synth={len(synthetic_only)}), Val={len(val_all)}, Test={len(test_all)}")

# ============================================================
# VERIFICATION: No synthetic in validation or test
# ============================================================
print("\n" + "="*60)
print("Step 7: Verifying synthetic images only in training")
print("="*60)

errors = 0
for fold in all_folds:
    synth_in_val = [img for img in fold['val'] if img.startswith("synth_")]
    synth_in_test = [img for img in fold['test'] if img.startswith("synth_")]

    if synth_in_val:
        print(f"❌ Fold {fold['fold']}: {len(synth_in_val)} synthetic in validation!")
        errors += 1
    if synth_in_test:
        print(f"❌ Fold {fold['fold']}: {len(synth_in_test)} synthetic in test!")
        errors += 1

if errors == 0:
    print("✅ No synthetic images in validation or test sets")
else:
    print(f"❌ {errors} errors found")
    exit()

# ============================================================
# VERIFICATION: No test image appears in multiple folds
# ============================================================
print("\n" + "="*60)
print("Step 8: Verifying no test image in multiple folds")
print("="*60)

all_test_images = []
for fold in all_folds:
    all_test_images.extend(fold['test'])

counts = Counter(all_test_images)
duplicates = [img for img, count in counts.items() if count > 1]

if duplicates:
    print(f"❌ ERROR: {len(duplicates)} images appear in multiple test folds!")
    print(f"   First 10: {duplicates[:10]}")
    exit()
else:
    print(f"✅ No duplicates across test folds")

# ============================================================
# VERIFICATION: No overlap within same fold
# ============================================================
print("\n" + "="*60)
print("Step 9: Verifying no overlap within folds")
print("="*60)

for fold in all_folds:
    train_set = set(fold['train'])
    val_set = set(fold['val'])
    test_set = set(fold['test'])

    train_val = train_set & val_set
    train_test = train_set & test_set
    val_test = val_set & test_set

    if train_val:
        print(f"❌ Fold {fold['fold']}: Train/Val overlap: {len(train_val)}")
    if train_test:
        print(f"❌ Fold {fold['fold']}: Train/Test overlap: {len(train_test)}")
    if val_test:
        print(f"❌ Fold {fold['fold']}: Val/Test overlap: {len(val_test)}")

print("✅ Overlap check complete")

# ============================================================
# SAVE SPLITS TO FILES
# ============================================================
print("\n" + "="*60)
print("Step 10: Saving splits to files")
print("="*60)

SPLITS_DIR = "/content/drive/MyDrive/fold_splits"
os.makedirs(SPLITS_DIR, exist_ok=True)

for fold in all_folds:
    with open(f"{SPLITS_DIR}/fold_{fold['fold']}_train.txt", 'w') as f:
        f.write("\n".join(fold['train']))
    with open(f"{SPLITS_DIR}/fold_{fold['fold']}_val.txt", 'w') as f:
        f.write("\n".join(fold['val']))
    with open(f"{SPLITS_DIR}/fold_{fold['fold']}_test.txt", 'w') as f:
        f.write("\n".join(fold['test']))

    print(f"Fold {fold['fold']}: saved")

print(f"\n✅ Splits saved to: {SPLITS_DIR}")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

for fold in all_folds:
    print(f"Fold {fold['fold']}: Train={len(fold['train'])}, Val={len(fold['val'])}, Test={len(fold['test'])}")

print("\n✅ Script 2 complete. Ready for Script 3.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Restoring data
✅ Data restored

Step 2: Collecting all original images
Original images: 1133

Step 3: Loading synthetic images
Synthetic images: 500

Step 4: Creating master dataset
Copying original images...
Copying synthetic images...

✅ Master dataset created with 1633 images

Step 5: Separating original and synthetic
Original images only: 1133
Synthetic images only: 500

Step 6: Generating 5-fold splits (original images only)
Fold 1: Train=1179 (orig=679 + synth=500), Val=227, Test=227
Fold 2: Train=1179 (orig=679 + synth=500), Val=227, Test=227
Fold 3: Train=1179 (orig=679 + synth=500), Val=227, Test=227
Fold 4: Train=1180 (orig=680 + synth=500), Val=227, Test=226
Fold 5: Train=1180 (orig=680 + synth=500), Val=227, Test=226

Step 7: Verifying synthetic images only in training
✅ No synthetic images in validation or test sets

Step 8: Verifying no 

In [7]:
"""
Script 3: Verify a Specific Fold
Run this before training to ensure no leakage
"""

import os
from google.colab import drive

# Configuration
FOLD_TO_CHECK = 1  # Change to 1, 2, 3, 4, or 5

drive.mount('/content/drive')

# ============================================================
# LOAD SPLITS
# ============================================================
print("="*60)
print(f"Verifying Fold {FOLD_TO_CHECK}")
print("="*60)

SPLITS_DIR = "/content/drive/MyDrive/fold_splits"

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_CHECK}_train.txt", 'r') as f:
    train_set = set(line.strip() for line in f if line.strip())

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_CHECK}_val.txt", 'r') as f:
    val_set = set(line.strip() for line in f if line.strip())

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_CHECK}_test.txt", 'r') as f:
    test_set = set(line.strip() for line in f if line.strip())

print(f"Train: {len(train_set)}")
print(f"Val: {len(val_set)}")
print(f"Test: {len(test_set)}")

# ============================================================
# CHECK OVERLAPS
# ============================================================
print("\n" + "="*60)
print("Checking overlaps")
print("="*60)

train_val = train_set & val_set
train_test = train_set & test_set
val_test = val_set & test_set

if train_val:
    print(f"❌ Train/Val overlap: {len(train_val)} images")
    print(f"   Examples: {list(train_val)[:5]}")
else:
    print("✅ No train/val overlap")

if train_test:
    print(f"❌ Train/Test overlap: {len(train_test)} images")
    print(f"   Examples: {list(train_test)[:5]}")
else:
    print("✅ No train/test overlap")

if val_test:
    print(f"❌ Val/Test overlap: {len(val_test)} images")
    print(f"   Examples: {list(val_test)[:5]}")
else:
    print("✅ No val/test overlap")

# ============================================================
# CHECK SYNTHETIC IMAGES
# ============================================================
print("\n" + "="*60)
print("Checking synthetic images")
print("="*60)

synth_in_val = [img for img in val_set if img.startswith("synth_")]
synth_in_test = [img for img in test_set if img.startswith("synth_")]

if synth_in_val:
    print(f"❌ {len(synth_in_val)} synthetic images in validation set!")
    print(f"   Examples: {synth_in_val[:5]}")
else:
    print("✅ No synthetic images in validation")

if synth_in_test:
    print(f"❌ {len(synth_in_test)} synthetic images in test set!")
    print(f"   Examples: {synth_in_test[:5]}")
else:
    print("✅ No synthetic images in test")

# ============================================================
# FINAL VERDICT
# ============================================================
print("\n" + "="*60)
if train_val or train_test or val_test or synth_in_val or synth_in_test:
    print("❌ VERIFICATION FAILED - Do not train this fold")
else:
    print("✅ VERIFICATION PASSED - Ready to train")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verifying Fold 1
Train: 1179
Val: 227
Test: 227

Checking overlaps
✅ No train/val overlap
✅ No train/test overlap
✅ No val/test overlap

Checking synthetic images
✅ No synthetic images in validation
✅ No synthetic images in test

✅ VERIFICATION PASSED - Ready to train


In [9]:
"""
Script 4: Train a Single Fold
Run after verification passes
"""

import os
import shutil
from ultralytics import YOLO
from google.colab import drive

# Configuration
FOLD_TO_TRAIN = 1  # Change to 1, 2, 3, 4, or 5

drive.mount('/content/drive')

# ============================================================
# RESTORE MASTER DATASET
# ============================================================
print("="*60)
print(f"Training Fold {FOLD_TO_TRAIN}")
print("="*60)

if not os.path.exists("/content/master_dataset"):
    if os.path.exists("/content/drive/MyDrive/master_dataset"):
        !cp -r "/content/drive/MyDrive/master_dataset" /content/master_dataset
    else:
        print("❌ Master dataset not found! Run Script 2 first.")
        exit()

print("✅ Master dataset restored")

# ============================================================
# LOAD SPLITS
# ============================================================
SPLITS_DIR = "/content/drive/MyDrive/fold_splits"

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_train.txt", 'r') as f:
    train_set = [line.strip() for line in f if line.strip()]

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_val.txt", 'r') as f:
    val_set = [line.strip() for line in f if line.strip()]

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_test.txt", 'r') as f:
    test_set = [line.strip() for line in f if line.strip()]

print(f"Train: {len(train_set)}")
print(f"Val: {len(val_set)}")
print(f"Test: {len(test_set)}")

# ============================================================
# CREATE FOLD DIRECTORY
# ============================================================
fold_dir = f"/content/fold_{FOLD_TO_TRAIN}"
if os.path.exists(fold_dir):
    shutil.rmtree(fold_dir)

train_img_dir = f"{fold_dir}/train/images"
train_label_dir = f"{fold_dir}/train/labels"
val_img_dir = f"{fold_dir}/val/images"
val_label_dir = f"{fold_dir}/val/labels"
test_img_dir = f"{fold_dir}/test/images"
test_label_dir = f"{fold_dir}/test/labels"

for d in [train_img_dir, train_label_dir, val_img_dir, val_label_dir, test_img_dir, test_label_dir]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# COPY FILES
# ============================================================
print("\nCopying files...")

def copy_files(images, dst_img_dir, dst_label_dir):
    for img in images:
        src_img = os.path.join("/content/master_dataset/images", img)
        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(dst_img_dir, img))
        label = img.replace('.jpg', '.txt').replace('.png', '.txt')
        src_label = os.path.join("/content/master_dataset/labels", label)
        if os.path.exists(src_label):
            shutil.copy(src_label, os.path.join(dst_label_dir, label))

copy_files(train_set, train_img_dir, train_label_dir)
copy_files(val_set, val_img_dir, val_label_dir)
copy_files(test_set, test_img_dir, test_label_dir)

print("✅ Files copied")

# ============================================================
# TRAIN
# ============================================================
print("\n" + "="*60)
print("Training...")
print("="*60)

data_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
nc: 1
names: ['Ganoderma']
"""
with open(f"{fold_dir}/data.yaml", 'w') as f:
    f.write(data_yaml)

model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{fold_dir}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=30,
    device=0,
    project=fold_dir,
    name="run",
    exist_ok=True,
    verbose=True
)

# ============================================================
# EVALUATE
# ============================================================
print("\n" + "="*60)
print("Evaluating on test set...")
print("="*60)

test_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
test: {test_img_dir}
nc: 1
names: ['Ganoderma']
"""
with open(f"{fold_dir}/test_data.yaml", 'w') as f:
    f.write(test_yaml)

test_results = model.val(data=f"{fold_dir}/test_data.yaml", split='test')

print("\n" + "="*60)
print(f"FOLD {FOLD_TO_TRAIN} RESULTS")
print("="*60)
print(f"mAP@50:     {test_results.box.map50:.4f}")
print(f"Precision:  {test_results.box.mp:.4f}")
print(f"Recall:     {test_results.box.mr:.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================
import pandas as pd
results_df = pd.DataFrame([{
    'fold': FOLD_TO_TRAIN,
    'mAP50': test_results.box.map50,
    'precision': test_results.box.mp,
    'recall': test_results.box.mr
}])
results_df.to_csv(f"/content/drive/MyDrive/fold_{FOLD_TO_TRAIN}_results.csv", index=False)

# Save model
shutil.copy(f"{fold_dir}/run/weights/best.pt", f"/content/drive/MyDrive/fold_{FOLD_TO_TRAIN}_model.pt")

print(f"\n✅ Results and model saved to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training Fold 1
✅ Master dataset restored
Train: 1179
Val: 227
Test: 227

Copying files...
✅ Files copied

Training...
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=30, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, 

In [ ]:
"""
Script 4: Train a Single Fold with Stronger Augmentations
Uses built-in YOLO augmentations (no synthetic data)
"""

import os
import shutil
from ultralytics import YOLO
from google.colab import drive

# Configuration
FOLD_TO_TRAIN = 1  # Change to 1, 2, 3, 4, or 5
EPOCHS = 100        # More epochs with augmentations
BATCH_SIZE = 50
IMGSZ = 640
DEVICE = 0

drive.mount('/content/drive')

# ============================================================
# RESTORE MASTER DATASET (Original only)
# ============================================================
print("="*60)
print(f"Training Fold {FOLD_TO_TRAIN} with Stronger Augmentations")
print("="*60)

# Use original dataset only (no synthetic)
if not os.path.exists("/content/ganoderma"):
    !cp -r "/content/drive/MyDrive/ganoderma" /content/ganoderma
print("✅ Dataset restored")

# ============================================================
# LOAD SPLITS (Original only, no synthetic)
# ============================================================
SPLITS_DIR = "/content/drive/MyDrive/fold_splits"

# Read original-only splits (these have no synthetic images)
with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_train.txt", 'r') as f:
    train_set = [line.strip() for line in f if line.strip() and not line.startswith("synth_")]

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_val.txt", 'r') as f:
    val_set = [line.strip() for line in f if line.strip() and not line.startswith("synth_")]

with open(f"{SPLITS_DIR}/fold_{FOLD_TO_TRAIN}_test.txt", 'r') as f:
    test_set = [line.strip() for line in f if line.strip() and not line.startswith("synth_")]

print(f"Train: {len(train_set)} (original only)")
print(f"Val: {len(val_set)}")
print(f"Test: {len(test_set)}")

# ============================================================
# CREATE FOLD DIRECTORY
# ============================================================
fold_dir = f"/content/fold_{FOLD_TO_TRAIN}_aug"
if os.path.exists(fold_dir):
    shutil.rmtree(fold_dir)

train_img_dir = f"{fold_dir}/train/images"
train_label_dir = f"{fold_dir}/train/labels"
val_img_dir = f"{fold_dir}/val/images"
val_label_dir = f"{fold_dir}/val/labels"
test_img_dir = f"{fold_dir}/test/images"
test_label_dir = f"{fold_dir}/test/labels"

for d in [train_img_dir, train_label_dir, val_img_dir, val_label_dir, test_img_dir, test_label_dir]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# COPY FILES
# ============================================================
print("\nCopying files...")

def copy_files(images, dst_img_dir, dst_label_dir):
    for img in images:
        # Find source
        for src in ["train", "valid", "test"]:
            src_img = f"/content/ganoderma/{src}/images/{img}"
            if os.path.exists(src_img):
                shutil.copy(src_img, os.path.join(dst_img_dir, img))
                break
        label = img.replace('.jpg', '.txt').replace('.png', '.txt')
        for src in ["train", "valid", "test"]:
            src_label = f"/content/ganoderma/{src}/labels/{label}"
            if os.path.exists(src_label):
                shutil.copy(src_label, os.path.join(dst_label_dir, label))
                break

copy_files(train_set, train_img_dir, train_label_dir)
copy_files(val_set, val_img_dir, val_label_dir)
copy_files(test_set, test_img_dir, test_label_dir)

print("✅ Files copied")

# ============================================================
# TRAIN WITH STRONGER AUGMENTATIONS
# ============================================================
print("\n" + "="*60)
print("Training with stronger augmentations:")
print("  - Rotation: ±15°")
print("  - Translation: ±20%")
print("  - Scaling: ±70%")
print("  - MixUp: 50%")
print("  - Copy-Paste: 50%")
print("  - Mosaic: 100%")
print("="*60)

data_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
nc: 1
names: ['Ganoderma']
"""
with open(f"{fold_dir}/data.yaml", 'w') as f:
    f.write(data_yaml)

model = YOLO('yolov8n.pt')

results = model.train(
    data=f"{fold_dir}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=fold_dir,
    name="run",
    exist_ok=True,
    verbose=True,

    # ============================================================
    # AUGMENTATION SETTINGS (Enhanced)
    # ============================================================
    hsv_h=0.02,        # Hue variation (default: 0.015)
    hsv_s=0.8,         # Saturation variation (default: 0.7)
    hsv_v=0.5,         # Brightness variation (default: 0.4)

    degrees=15,        # Rotation ±15° (default: 0)
    translate=0.2,     # Translation ±20% (default: 0.1)
    scale=0.7,         # Scaling ±70% (default: 0.5)
    fliplr=0.5,        # Horizontal flip 50% (default: 0.5)
    flipud=0.1,        # Vertical flip 10% (default: 0)

    mosaic=1.0,        # Mosaic augmentation (default: 1.0)
    mixup=0.5,         # MixUp augmentation (default: 0)
    copy_paste=0.5,    # Copy-Paste augmentation (default: 0)
)

# ============================================================
# EVALUATE ON TEST SET
# ============================================================
print("\n" + "="*60)
print("Evaluating on test set...")
print("="*60)

test_yaml = f"""
train: {train_img_dir}
val: {val_img_dir}
test: {test_img_dir}
nc: 1
names: ['Ganoderma']
"""
with open(f"{fold_dir}/test_data.yaml", 'w') as f:
    f.write(test_yaml)

test_results = model.val(data=f"{fold_dir}/test_data.yaml", split='test')

print("\n" + "="*60)
print(f"FOLD {FOLD_TO_TRAIN} RESULTS (Stronger Augmentations)")
print("="*60)
print(f"mAP@50:     {test_results.box.map50:.4f}")
print(f"Precision:  {test_results.box.mp:.4f}")
print(f"Recall:     {test_results.box.mr:.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================
import pandas as pd
results_df = pd.DataFrame([{
    'fold': FOLD_TO_TRAIN,
    'mAP50': test_results.box.map50,
    'precision': test_results.box.mp,
    'recall': test_results.box.mr,
    'augmentations': 'strong'
}])
results_df.to_csv(f"/content/drive/MyDrive/fold_{FOLD_TO_TRAIN}_aug_results.csv", index=False)

# Save model
shutil.copy(f"{fold_dir}/run/weights/best.pt", f"/content/drive/MyDrive/fold_{FOLD_TO_TRAIN}_aug_model.pt")

print(f"\n✅ Results and model saved to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training Fold 1 with Stronger Augmentations
✅ Dataset restored
Train: 679 (original only)
Val: 227
Test: 227

Copying files...
✅ Files copied

Training with stronger augmentations:
  - Rotation: ±15°
  - Translation: ±20%
  - Scaling: ±70%
  - MixUp: 50%
  - Copy-Paste: 50%
  - Mosaic: 100%
New https://pypi.org/project/ultralytics/8.4.58 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=50, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fold_1_aug/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropou